In [1]:
import os
import time
import tomllib
import sqlparse
import warnings
import pymysql
import numpy as np
import pandas as pd
import datetime as dt
from tqdm import tqdm
from dotenv import load_dotenv

# Vectorization vs Parallelization

For June 2026, the seismic revision routine makes a search of 17 different checks on the seismic data. Each check is a function that takes in the seismic data and performs some operations on it to check for certain conditions. The checks are performed sequentially, which means that each check is performed one after the other. This can be time-consuming, especially if the seismic data is large. In order to handle this, initially parallelization was implemented using the multiprocessing library in Python. This allowed the checks to be performed in parallel, which significantly reduced the time taken to perform the checks. However, this approach had some limitations, such as the overhead of creating and managing multiple processes, and the need to ensure that the checks were thread-safe.

Furthermore, there are some operations that are performed on the seismic data that can be vectorized using libraries such as NumPy. Vectorization allows for the operations to be performed on entire arrays of data at once, rather than iterating through each element individually. This can significantly reduce the time taken to perform the operations, as it takes advantage of the underlying hardware optimizations for array operations. In addition, there is no need to initialize multiple workers or manage the overhead associated with parallelization. By using vectorization, we can reduce energy consumption and improve the efficiency of the seismic revision routine, while also simplifying the code and reducing the potential for errors. Overall, while parallelization can be useful in certain situations, we will check if vectorization is a more efficient and effective approach for handling large datasets and performing complex operations on them.

In this notebook, we will compare the performance of vectorization and parallelization for a specific check in the seismic revision routine. We will implement both approaches and measure the time taken to perform the check on a sample seismic dataset. We will also analyze the results and discuss the advantages and disadvantages of each approach. Finally, we will make recommendations on which approach to use for different scenarios in the seismic revision routine. The main idea is to define single functions to perform each check, and then use either vectorization or parallelization to apply those functions to the seismic data.

## Query seismic data from database

Let's start by querying the seismic data from the database. We will use the `pymysql` library to connect to the database and execute a SQL query to retrieve the seismic data. We will also use the `dotenv` library to load the database credentials from a `.env` file, as stated in the previous notebook.

In [2]:
load_dotenv(dotenv_path=os.path.join(os.getcwd(), '.env'))

# Cutover date: SC3 → SC6
SC6_CUTOVER = dt.datetime(2026, 3, 17, 0, 0, 0)

def _build_connection(prefix: str):
    """Create a pymysql connection using .env credentials for a given prefix."""
    return pymysql.connect(
        host=os.getenv(f'SERVER_{prefix}_HOST'),
        port=int(os.getenv(f'SERVER_{prefix}_PORT', 3306)),
        user=os.getenv(f'SERVER_{prefix}_USERNAME'),
        password=os.getenv(f'SERVER_{prefix}_PASSWORD'),
        db=os.getenv(f'SERVER_{prefix}_DATABASE'),
    )


def _query_db(
    prefix: str,
    query: str,
    start_time: dt.datetime,
    end_time: dt.datetime,
    desc: str,
    **kwargs,
) -> pd.DataFrame:
    """
    Execute a time-bounded SQL query against a single database.

    Parameters
    ----------
    prefix : str
        Credential prefix — 'SC3' or 'SC6'.
    query : str
        Base SQL query ending before the BETWEEN clause.
    start_time, end_time : datetime
        Time bounds for the query.
    desc : str
        Label shown in the tqdm progress bar.
    """
    start_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
    end_str   = end_time.strftime("%Y-%m-%d %H:%M:%S")
    full_query = (
        f"{query} '{start_str}' AND '{end_str}' "
        f"ORDER BY Origin.time_value ASC;"
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        conn = _build_connection(prefix)
        try:
            with tqdm(
                total=1,
                desc=desc,
                unit="query",
                leave=False,
                bar_format="{desc}",
            ) as pbar:
                df = pd.read_sql_query(full_query, conn, **kwargs)
                pbar.update(1)
        finally:
            conn.close()

    return df

def _to_naive_utc(t: dt.datetime) -> dt.datetime:
    """
    Normalize a datetime to naive UTC.
    - Timezone-aware → convert to UTC, strip tzinfo
    - Naive → assumed UTC already, returned as-is
    """
    if t.tzinfo is not None:
        return t.astimezone(dt.timezone.utc).replace(tzinfo=None)
    return t

def connect_to_db(
    query: str,
    start_time: dt.datetime = None,
    end_time: dt.datetime = None,
    **kwargs,
) -> pd.DataFrame:
    """
    Query SC3 and/or SC6 databases depending on the requested time range.

    Decision logic:
        - end_time   <= SC6_CUTOVER  → SC3 only
        - start_time >= SC6_CUTOVER  → SC6 only
        - start_time <  SC6_CUTOVER  < end_time → both (with overlap warning)

    Parameters
    ----------
    query : str
        Base SQL query ending before the BETWEEN clause.
    start_time : datetime, optional
        Start of the time range (UTC). If None, runs the query as-is.
    end_time : datetime, optional
        End of the time range (UTC). If None, runs the query as-is.

    Returns
    -------
    pd.DataFrame
        Query results, merged and sorted by time_value when both DBs are hit.
    """
    # --- No time range: run query as-is against SC3 (legacy default) ----------
    if start_time is None or end_time is None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            conn = _build_connection("SC3")
            try:
                with tqdm(
                    total=1,
                    desc="Querying database...",
                    unit="query",
                    leave=False,
                    bar_format="{desc}",
                ) as pbar:
                    df = pd.read_sql_query(query, conn, **kwargs)
                    pbar.update(1)
            finally:
                conn.close()
        return df

    # --- Normalize to naive UTC before any comparison ------------------------
    start_time = _to_naive_utc(start_time)
    end_time   = _to_naive_utc(end_time)

    # --- Determine which databases are needed --------------------------------
    only_sc3 = end_time   <= SC6_CUTOVER
    only_sc6 = start_time >= SC6_CUTOVER
    both     = not only_sc3 and not only_sc6       # straddles the cutover

    if both:
        warnings.warn(
            f"\n[DATABASE WARNING] The requested time range "
            f"({start_time:%Y-%m-%d %H:%M:%S} → {end_time:%Y-%m-%d %H:%M:%S}) "
            f"spans the seiscomp3-seiscomp6 database cutover ({SC6_CUTOVER:%Y-%m-%d %H:%M:%S} UTC). "
            f"Both databases will be queried and results merged.\n",
            UserWarning,
            stacklevel=2,
        )

    # --- SC3 only ------------------------------------------------------------
    if only_sc3:
        return _query_db(
            prefix="SC3",
            query=query,
            start_time=start_time,
            end_time=end_time,
            desc="Querying SC3 database...",
            **kwargs,
        )

    # --- SC6 only ------------------------------------------------------------
    if only_sc6:
        return _query_db(
            prefix="SC6",
            query=query,
            start_time=start_time,
            end_time=end_time,
            desc="Querying SC6 database...",
            **kwargs,
        )

    # --- Both databases (straddles cutover) ----------------------------------
    # SC3: [start_time, SC6_CUTOVER)
    # SC6: [SC6_CUTOVER, end_time]
    df_sc3 = _query_db(
        prefix="SC3",
        query=query,
        start_time=start_time,
        end_time=SC6_CUTOVER,
        desc="Querying SC3 database (1/2)...",
        **kwargs,
    )
    df_sc6 = _query_db(
        prefix="SC6",
        query=query,
        start_time=SC6_CUTOVER,
        end_time=end_time,
        desc="Querying SC6 database (2/2)...",
        **kwargs,
    )

    # Merge and re-sort by time_value
    df = (
        pd.concat([df_sc3, df_sc6], ignore_index=True)
        .sort_values("time_value")
        .reset_index(drop=True)
    )

    return df

In [3]:
# Read revision.sql file
with open('./queries/revision.sql', 'r') as file:
    revision_query = file.read()

clean_sql = sqlparse.format(revision_query, strip_comments=True).strip()

initial_time = dt.datetime(2023, 1, 1, 0, 0, 0)
final_time = dt.datetime.now(dt.UTC)
event_df3 = connect_to_db(clean_sql, start_time=initial_time, end_time=final_time)

print(f"Number of rows in the seismic data: {len(event_df3)}")
# Print how events are by event_type
print("Number of events by event_type:")
print(event_df3['event_type'].value_counts())

/tmp/ipykernel_8763/1579909875.py:9: UserWarning: 
[DATABASE WARNING] The requested time range (2023-01-01 00:00:00 → 2026-06-11 09:21:39) spans the seiscomp3-seiscomp6 database cutover (2026-03-17 00:00:00 UTC). Both databases will be queried and results merged.

  event_df3 = connect_to_db(clean_sql, start_time=initial_time, end_time=final_time)


Number of rows in the seismic data: 247486
Number of events by event_type:
event_type
not locatable                  137901
earthquake                      93333
not existing                     5469
explosion                        5258
outside of network interest      4935
volcanic eruption                 505
induced earthquake                  4
other                               1
Name: count, dtype: int64


## 1. Comparison checks

The seismic revision routine performs a list of quality checks on earthquakes, based on relational or absolute thresholds. These checks are designed to identify earthquakes that may not be reliable and may require further investigation. Some of the checks that are performed include:

1. High RMS: This check identifies earthquakes with a high root-mean-square (RMS) value, which indicates that the seismic data is noisy and may not be reliable. The threshold for this check is typically set at a certain value, such as 1.51.
2. Localization uncertainty: This check identifies earthquakes with a high localization uncertainty, which indicates that the location of the earthquake is not well-defined. The threshold for this check is typically set at a certain value, such as 12 km. It is applied both on latitude, longitude, and depth.
3. Depth check: This check identifies earthquakes with a depth that is outside of a certain range, such as between 0 and 700 km. This check is important because earthquakes that are too shallow or too deep may not be reliable and may require further investigation.

For all these type of checks, it is possible to vectorize the solution by applying the check to the entire dataset at once, rather than iterating through each earthquake individually. The idea here is to create a single general function, receiving the filtered seismic data, the threshold value or values (if there are multiple thresholds), and the column to be checked. The function will then apply the check to the entire dataset and return a boolean mask indicating which earthquakes meet the criteria for being flagged as unreliable. This approach can significantly reduce the time taken to perform the checks, without making it too complicated to be debugged or maintained.

In [4]:
# Previous version
def single_check(event):
    observations = []

    # First check: High RMS values
    exceptions = ["not locatable", "outside of network interest", "volcanic eruption", "explosion", "not existing"]
    if event['quality_standardError'] > 1.51 and event['event_type'] not in exceptions:
        observations.append("High RMS value")

    if len(observations) > 0:  # If the event has observations, return the information
        return event, observations
    else:
        return None, None

# For loop version
time1 = time.time()
results = []
for _, event in event_df3.iterrows():
    result, obs = single_check(event)
    if result is not None:
        results.append((result, obs))
high_rms_df_loop = pd.DataFrame([res[0] for res in results])
time2 = time.time()
print(f"Number of events with high RMS (loop version): {len(high_rms_df_loop)}")
print(f"Time taken for high RMS check (loop version): {time2 - time1:.4f} seconds")

Number of events with high RMS (loop version): 31
Time taken for high RMS check (loop version): 13.0590 seconds


In [5]:
# Vectorized function to make comparison between a column and a threshold value
def build_quality_mask(
    events: pd.DataFrame,
    column: str,
    mode: str,
    threshold=None,
    lower=None,
    upper=None,
    dtype=np.float64
) -> np.ndarray:
    """
    Vectorized generic comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    column : str
        Column to evaluate.
    mode : str
        Comparison mode:
            'gt'       -> values > threshold
            'ge'       -> values >= threshold
            'lt'       -> values < threshold
            'le'       -> values <= threshold
            'between'  -> lower <= values <= upper
            'outside'  -> values < lower or values > upper
            'abs_gt'   -> abs(values) > threshold
            'abs_ge'   -> abs(values) >= threshold
    threshold : float, optional
        Threshold for one-sided comparisons.
    lower, upper : float, optional
        Bounds for range comparisons.
    dtype : numpy dtype
        Target dtype for NumPy conversion.

    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    values = events[column].to_numpy(dtype=dtype, copy=False)

    if mode == 'gt':
        return values > threshold
    elif mode == 'ge':
        return values >= threshold
    elif mode == 'lt':
        return values < threshold
    elif mode == 'le':
        return values <= threshold
    elif mode == 'between':
        return (values >= lower) & (values <= upper)
    elif mode == 'outside':
        return (values < lower) | (values > upper)
    elif mode == 'abs_gt':
        return np.abs(values) > threshold
    elif mode == 'abs_ge':
        return np.abs(values) >= threshold
    else:
        raise ValueError(f"Unsupported mode: {mode}")

# Check for high RMS values on 'earthquake' and 'volcanic eruption' event types
time1 = time.time()
rms_threshold = 1.51
rms_mask = build_quality_mask(
    events=event_df3[event_df3['event_type'].isin(['earthquake', 'volcanic eruption'])],
    column='quality_standardError',
    mode='gt',
    threshold=rms_threshold
)
high_rms_df = event_df3[event_df3['event_type'].isin(['earthquake', 'volcanic eruption'])][rms_mask]
time2 = time.time()
print(f"Number of events with high RMS: {len(high_rms_df)}")
print(f"Time taken for high RMS check: {time2 - time1:.4f} seconds")

Number of events with high RMS: 29
Time taken for high RMS check: 0.1104 seconds


As you can see, the vectorized version of the high RMS check is significantly faster than the loop version (13.53 s to just 0.0812 s!). This is because the vectorized version takes advantage of NumPy's optimized array operations, which are implemented in C and can be executed much faster than Python loops. In contrast, the loop version iterates through each event one by one, which is much slower, especially for large datasets. Additionally, the vectorized version is more concise and easier to read, as it eliminates the need for explicit loops and conditional statements. Overall, this demonstrates the significant performance benefits of using vectorization for data processing tasks in Python.

Now let's create a wrapper function to apply multiple checks at once:

In [6]:
# Wrapper function to apply multiple checks at once
def seismic_quality_checks(events: pd.DataFrame) -> pd.DataFrame:
    """
    Apply common earthquake quality checks and return flagged events.
    """
    selections = events[events['event_type'].eq('earthquake')].reset_index(drop=True)

    if selections.empty:
        return selections.iloc[0:0].copy()

    masks = {
        'High RMS': build_quality_mask(
            selections, column='quality_standardError', mode='gt', threshold=1.51
        ),
        'High err_lat': build_quality_mask(
             selections, column='latitude_uncertainty', mode='gt', threshold=12.0
        ),
        'High err_lon': build_quality_mask(
            selections, column='longitude_uncertainty', mode='gt', threshold=12.0
        ),
        'High err_depth': build_quality_mask(
            selections, column='depth_uncertainty', mode='gt', threshold=12.0
        ),
        'Invalid depth': build_quality_mask(
            selections, column='depth_value', mode='outside', lower=0.0, upper=200.0
        ),
        'Earthquake with 6 or less phase count' : build_quality_mask(
            selections, column='quality_associatedPhaseCount', mode='le', threshold=6.0
        )
    }

    combined_mask = np.zeros(len(selections), dtype=bool)
    for mask in masks.values():
        combined_mask |= mask

    flagged = selections.loc[combined_mask].copy()

    flagged_idx = np.where(combined_mask)[0]
    observations = []
    for i in flagged_idx:
        obs = [name for name, mask in masks.items() if mask[i]]
        observations.append(', '.join(obs))

    flagged['Observations'] = observations
    return flagged.reset_index(drop=True)

# Apply the checks and measure time
time1 = time.time()
flagged_events = seismic_quality_checks(event_df3)
time2 = time.time()
print(f"Number of flagged events: {len(flagged_events)}")
print(f"Time taken for seismic quality checks: {time2 - time1:.4f} seconds")

Number of flagged events: 644
Time taken for seismic quality checks: 0.1105 seconds


The wrapper function `seismic_quality_checks` applies multiple quality checks to the seismic data and returns a DataFrame of flagged events along with the observations for each event. The function first filters the input DataFrame to include only earthquake events, and then applies each check using the `build_quality_mask` function. The results of all checks are combined into a single boolean mask, which is used to select the flagged events. Finally, the observations for each flagged event are compiled into a new column in the resulting DataFrame.

Now, it is time to add more checks to the wrapper function in order to make it more comprehensive. To maintain the readability and simplicity of the code, we will use a TOML file to store the configuration for each check, including the column and event types to be checked, the mode of comparison, and the threshold values. This way, we can easily add or modify checks without having to change the code of the wrapper function itself. The wrapper function will read the configuration from the TOML file and apply the checks accordingly. This approach allows us to keep the code clean and maintainable while still providing a flexible way to manage the quality checks for seismic events.

In [7]:
# Read TOML file without comments
with open('./seismic_checks.toml', 'rb') as f:
    checks_config = tomllib.load(f)["checks"]

checks_config

[{'name': 'High RMS',
  'column': 'quality_standardError',
  'mode': 'ge',
  'threshold': 1.5,
  'event_type': ['earthquake', 'explosion', 'volcanic eruption']},
 {'name': 'High Latitude Uncertainty',
  'column': 'latitude_uncertainty',
  'mode': 'gt',
  'threshold': 12,
  'event_type': ['earthquake', 'explosion', 'volcanic eruption']},
 {'name': 'High Longitude Uncertainty',
  'column': 'longitude_uncertainty',
  'mode': 'gt',
  'threshold': 12,
  'event_type': ['earthquake', 'explosion', 'volcanic eruption']},
 {'name': 'High Depth Uncertainty',
  'column': 'depth_uncertainty',
  'mode': 'gt',
  'threshold': 12,
  'event_type': ['earthquake', 'volcanic eruption']},
 {'name': 'Negative Depth',
  'column': 'depth_value',
  'mode': 'lt',
  'threshold': 0,
  'event_type': ['earthquake',
   'explosion',
   'volcanic eruption',
   'not locatable',
   'outside of network interest']},
 {'name': 'Noncommon High Depth',
  'column': 'depth_value',
  'mode': 'ge',
  'threshold': 200,
  'event_ty

In [8]:
def load_checks(path: str = "seismic_checks.toml") -> list[dict]:
    """Load quality checks config from a TOML file."""
    with open(path, "rb") as f:
        return tomllib.load(f)["checks"]


def seismic_quality_checks(
    events: pd.DataFrame,
    checks_path: str = "seismic_checks.toml",
) -> pd.DataFrame:
    """
    Apply seismic quality checks loaded from a TOML config file.

    Each check specifies its own target event_type list, so different checks
    can apply to different subsets of the dataset. The special keyword "all"
    means the check applies to every event type present in the data.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    checks_path : str
        Path to the TOML config file.

    Returns
    -------
    pd.DataFrame
        Flagged events with an 'Observations' column listing all triggered
        checks per event. Each publicID appears at most once.
    """
    checks = load_checks(checks_path)

    if events.empty:
        return events.iloc[0:0].copy()

    # observations_map: { row_index -> [check_name, ...] }
    observations_map: dict[int, list[str]] = {}

    for check in checks:
        # Select only rows matching this check's event types
        subset = events[events["event_type"].isin(check["event_type"])]

        if subset.empty:
            continue

        # Build the quality mask on the subset using original index
        kwargs = {k: v for k, v in check.items() if k not in ("name", "event_type")}
        flagged_mask = build_quality_mask(subset, **kwargs)

        # Map triggered rows back to original DataFrame index
        flagged_original_idx = subset.index[flagged_mask]
        for idx in flagged_original_idx:
            observations_map.setdefault(idx, []).append(check["name"])

    if not observations_map:
        return events.iloc[0:0].copy()

    # Build output from all flagged original indices — each event appears once
    flagged_idx = sorted(observations_map.keys())
    flagged = events.loc[flagged_idx].copy()
    flagged["Observations"] = [
        ", ".join(observations_map[i]) for i in flagged_idx
    ]

    return flagged.reset_index(drop=True)

In [9]:
time1 = time.time()
flagged = seismic_quality_checks(event_df3)
time2 = time.time()
print(f"Number of flagged events: {len(flagged)}")
print(f"Time taken for seismic quality checks with TOML config: {time2 - time1:.4f} seconds")

Number of flagged events: 3418
Time taken for seismic quality checks with TOML config: 0.4799 seconds


In [10]:
flagged

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment,Observations
0,2023-01-01 15:06:03,SGC2023abeaub,-2.187500,NaN,0.535239,30.800264,102.311892,94.038259,4.0,4.0,...,not locatable,SGC,"Rionegro - Santander, Colombia",7.454178,-73.323887,None,NonLinLoc,Poveda_et_al_2018,None,Negative Depth
1,2023-01-02 13:08:57,SGC2023acvwti,-0.840000,NaN,0.180000,2.000000,2.404163,2.404163,7.0,7.0,...,not locatable,SGC,"Casabianca - Tolima, Colombia",4.944333,-75.330500,None,Hypo71,RSNC,None,Negative Depth
2,2023-01-02 17:15:18,SGC2023adebaw,0.000000,1.357507,0.348863,0.000000,5.439274,13.657190,5.0,5.0,...,explosion,SGC,"El Paso - Cesar, Colombia",9.683134,-73.647324,MLr_4,LOCSAT,iasp91,None,High Longitude Uncertainty
3,2023-01-03 17:29:10,SGC2023afafwm,-0.960000,1.132685,0.070000,NaN,NaN,NaN,6.0,6.0,...,explosion,SGC,"La Jagua de Ibirico - Cesar, Colombia",9.489000,-73.477833,M,Hypo71,RSNC,None,Negative Depth
4,2023-01-03 17:37:48,SGC2023afanhs,-0.280000,1.001067,0.220000,3.600000,1.697056,1.697056,6.0,6.0,...,explosion,SGC,"El Paso - Cesar, Colombia",9.624000,-73.521667,M,Hypo71,RSNC,None,Negative Depth
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3413,2026-06-07 19:08:52,SGC2026ldrces,10.000000,1.534450,2.435437,NaN,NaN,NaN,NaN,NaN,...,explosion,SGC,"Barrancas - la Guajira, Colombia",11.020000,-72.881600,MLr_4,,,None,High RMS
3414,2026-06-07 19:09:19,SGC2026ldrcow,0.000000,1.674896,0.190000,5.000000,18.243355,18.243355,5.0,5.0,...,explosion,SGC,"Riohacha - la Guajira, Colombia",11.148500,-72.795000,MLr_4,Hypo71,CARMA,None,"High Latitude Uncertainty, High Longitude Unce..."
3415,2026-06-09 15:41:32,SGC2026lhbrho,-0.010000,1.737759,0.380000,6.700000,11.101576,11.101576,6.0,6.0,...,not locatable,SGC,"San Antonio - Tolima, Colombia",3.966000,-75.573833,MLr_2,Hypo71,RSNC,None,Negative Depth
3416,2026-06-10 05:10:36,SGC2026licmkc,0.000000,4.199263,1.571618,0.000000,3.538102,4.498088,29.0,25.0,...,earthquake,SGC,Océano Pacífico,3.351564,-82.589287,mb,LOCSAT,iasp91,None,High RMS


In [11]:
# Filter events from 2026-03-01 to 2026-03-17
march_events = event_df3[
    (event_df3['time_value'] >= dt.datetime(2026, 3, 1)) &
    (event_df3['time_value'] <= dt.datetime(2026, 3, 17))
].copy()
march_events

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment
223623,2026-03-01 00:09:53,SGC2026eeieln,10.000000,0.683220,2.478282,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"MurindÃ³ - Antioquia, Colombia",6.989500,-76.697100,MLr_1,,,None
223624,2026-03-01 00:18:00,SGC2026eeilng,5.000000,1.743837,0.090000,NaN,NaN,NaN,5.0,5.0,...,NaN,not locatable,SGC,"Riosucio - ChocÃ³, Colombia",7.213833,-77.126000,MLr_1,Hypo71,RSNC,None
223625,2026-03-01 00:19:42,SGC2026eeimxl,10.000000,-0.088892,3.748619,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
223626,2026-03-01 00:22:47,SGC2026eeippb,12.226562,1.045598,0.253937,4.556525,7.031040,2.865880,8.0,8.0,...,NaN,earthquake,SGC,"Dabeiba - Antioquia, Colombia",6.991690,-76.253300,MLr_1,NonLinLoc,Poveda_et_al_2018,None
223627,2026-03-01 00:23:23,SGC2026eeiqbz,10.000000,0.033025,1.093982,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
227103,2026-03-16 23:18:05,SGC2026fhqbxv,103.632812,1.727774,0.597553,8.010460,4.339104,4.863137,22.0,22.0,...,NaN,earthquake,SGC,"Pauna - BoyacÃ¡, Colombia",5.615868,-73.956240,MLr_3,NonLinLoc,Poveda_et_al_2018,None
227104,2026-03-16 23:31:36,SGC2026fhqnnx,10.000000,-0.143127,13.207386,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
227105,2026-03-16 23:35:30,SGC2026fhqqxh,10.000000,-0.012095,223.843471,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
227106,2026-03-16 23:38:32,SGC2026fhqtng,132.109375,1.907054,0.491379,6.815534,4.893524,5.976666,27.0,26.0,...,NaN,earthquake,SGC,"JordÃ¡n - Santander, Colombia",6.698534,-73.124235,MLr_3,NonLinLoc,Poveda_et_al_2018,None


In [12]:
# Make checks for those events and see how many are flagged
time1 = time.time()
flagged_march = seismic_quality_checks(march_events)
time2 = time.time()
print(f"Number of flagged events in March 2026: {len(flagged_march)}")
print(f"Time taken for seismic quality checks on March 2026 events: {time2 - time1:.4f} seconds")

Number of flagged events in March 2026: 14
Time taken for seismic quality checks on March 2026 events: 0.0239 seconds


In [13]:
flagged_march[['time_value', 'publicID', 'event_type', 'text', 'Observations']]

,time_value,publicID,event_type,text,Observations
0,2026-03-07 03:13:02,SGC2026epopop,not locatable,"Tesalia - Huila, Colombia",Negative Depth
1,2026-03-11 17:28:59,SGC2026exzwrg,explosion,"La Jagua de Ibirico - Cesar, Colombia","High Latitude Uncertainty, High Longitude Unce..."
2,2026-03-12 17:30:29,SGC2026ezvqvs,explosion,"AgustÃ­n Codazzi - Cesar, Colombia",High RMS
3,2026-03-12 17:39:12,SGC2026ezvyjf,not locatable,"Murillo - Tolima, Colombia",Negative Depth
4,2026-03-12 19:04:54,SGC2026ezyufi,explosion,"Riohacha - La Guajira, Colombia","High RMS, High Latitude Uncertainty, High Long..."
5,2026-03-12 19:05:31,SGC2026ezyusv,explosion,"Barrancas - La Guajira, Colombia",High RMS
6,2026-03-13 18:08:51,SGC2026fbsqud,explosion,"Barrancas - La Guajira, Colombia",High RMS
7,2026-03-14 17:27:46,SGC2026fdnagp,explosion,"Becerrill - Cesar, Colombia",High RMS
8,2026-03-14 17:29:55,SGC2026fdncda,explosion,"ChiriguanÃ¡ - Cesar, Colombia",High RMS
9,2026-03-15 17:30:01,SGC2026ffivca,explosion,"El Paso - Cesar, Colombia",Negative Depth


In [14]:
subset = event_df3[event_df3["event_type"].isin(['earthquake'])]   # Check earthquakes with RMS = 0.0? ASK
subset

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment
0,2023-01-01 00:04:23,SGC2023aaaduv,9.355469,1.902402,0.951364,3.270166,1.939846,1.667912,58.0,56.0,...,NaN,earthquake,SGC,"Mesetas - Meta, Colombia",3.460591,-74.178675,MLr_3,NonLinLoc,Poveda_et_al_2018,None
3,2023-01-01 00:45:28,SGC2023aabnex,22.773438,2.467139,1.013426,2.863049,2.649246,3.378211,66.0,56.0,...,NaN,earthquake,SGC,"NunchÃ­a - Casanare, Colombia",5.457120,-72.123158,MLr_3,NonLinLoc,Poveda_et_al_2018,None
8,2023-01-01 01:01:53,SGC2023aacbji,44.335938,1.197228,0.665473,3.228578,2.669464,3.103757,26.0,26.0,...,NaN,earthquake,SGC,"Obando - Valle del Cauca, Colombia",4.563894,-75.921527,MLr_2,NonLinLoc,Poveda_et_al_2018,None
9,2023-01-01 01:06:17,SGC2023aacfdn,16.914062,1.339508,0.954153,5.316745,2.928134,3.695415,32.0,32.0,...,NaN,earthquake,SGC,"Mesetas - Meta, Colombia",3.484403,-74.249541,MLr_3,NonLinLoc,Poveda_et_al_2018,None
12,2023-01-01 01:28:29,SGC2023aacygo,33.671875,1.605881,1.193405,10.968184,4.216096,7.712120,20.0,20.0,...,NaN,earthquake,SGC,"Buenaventura - Valle del Cauca, Colombia",3.802959,-77.087637,MLr_1,NonLinLoc,Poveda_et_al_2018,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247476,2026-06-11 04:27:19,SGC2026ljwubl,122.757812,1.498522,0.829268,11.233050,5.479986,9.841625,17.0,17.0,...,NaN,earthquake,SGC,"Los Santos - Santander, Colombia",6.832615,-73.051413,MLr_3,NonLinLoc,Poveda_et_al_2018,None
247477,2026-06-11 04:42:48,SGC2026ljxhkq,24.917969,1.596724,0.755181,3.873772,1.990650,1.715827,36.0,35.0,...,NaN,earthquake,SGC,"Acevedo - Huila, Colombia",1.814658,-75.869005,MLr_2,NonLinLoc,Poveda_et_al_2018,None
247479,2026-06-11 04:53:08,SGC2026ljxqin,120.890000,1.520557,0.810000,8.300000,3.181981,3.181981,12.0,12.0,...,NaN,earthquake,SGC,"Ubaté - Cundinamarca, Colombia",5.342667,-73.834333,MLr_3,Hypo71,RSNC,None
247482,2026-06-11 05:26:18,SGC2026ljysxd,29.000000,3.022460,1.110000,4.200000,1.838478,1.838478,52.0,46.0,...,NaN,earthquake,SGC,Océano Pacífico,4.707000,-77.664167,MLr_1,Hypo71,RSNC,None


### Appendix: Locatable events

A check of the seismic revision routine is to identify events that are locatable, but are labeled incorrectly as "not locatable". At RSNC, a event is locatable if it has at least 4 p phases and 2 s phases associated to it. This check is important because it can help to identify earthquakes that may have been misclassified and may require further investigation.

However, the previous routine only checks a verification on the _'quality_associatedPhaseCount'_ column to be greater than or equal to 8 (plus one due to an event with 6 phases in seiscomp will have a _quality_associatedPhaseCount_ of 7). Then, an event with for example 8 p phases and 0 s phases would be flagged as locatable, which is not correct.

The challenge here is that query the number of p and s phases associated to each event requires a join between the _Origin_ and _Arrival_ tables in the database, which can be time-consuming, especially for large datasets. Additionally, the check needs to be performed for each event individually, which can further increase the time taken to perform the check. Therefore, the strategy here is to use the columns _quality_associatedPhaseCount_, _quality_usedPhaseCount_, _quality_usedStationCount_ and _quality_associatedStationCount_ to create a vectorized check that can identify locatable events without the need for a join between the tables. This approach can significantly reduce the time taken to perform the check, while still providing accurate results.

### Seiscomp3

In [15]:
# First example: 3 p and 3 s event (within 2026-03-01 11:13:00 and 2026-03-01 11:14:00)
start_filter = dt.datetime(2026, 3, 1, 11, 13, 0)
end_filter = dt.datetime(2026, 3, 1, 11, 14, 0)
subset_df = event_df3[(event_df3['time_value'] >= start_filter) & (event_df3['time_value'] <= end_filter)].copy()
subset_df[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
223750,2026-03-01 11:13:54,not locatable,7.0,7.0,6.0,NaN


In [16]:
# Second example: 4 p and 3 s event (within 2026-03-01 06:38:00 and 2026-03-01 06:39:00)
start_filter_2 = dt.datetime(2026, 3, 1, 6, 38, 0)
end_filter_2 = dt.datetime(2026, 3, 1, 6, 39, 0)
subset_df_2 = event_df3[(event_df3['time_value'] >= start_filter_2) & (event_df3['time_value'] <= end_filter_2)].copy()
subset_df_2[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
223695,2026-03-01 06:38:24,earthquake,8.0,8.0,7.0,NaN


In [17]:
# Third example: 4 p and 4 s event (within 2026-03-01 00:22:00 and 2026-03-01 00:23:00)
start_filter_3 = dt.datetime(2026, 3, 1, 0, 22, 0)
end_filter_3 = dt.datetime(2026, 3, 1, 0, 23, 0)
subset_df_3 = event_df3[(event_df3['time_value'] >= start_filter_3) & (event_df3['time_value'] <= end_filter_3)].copy()
subset_df_3[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
223626,2026-03-01 00:22:47,earthquake,8.0,8.0,4.0,NaN


In [18]:
# Fourth example: 4 p and 4 s event (within 2026-03-01 11:33:00 and 2026-03-01 11:34:00)
start_filter_4 = dt.datetime(2026, 3, 1, 11, 33, 0)
end_filter_4 = dt.datetime(2026, 3, 1, 11, 34, 0)
subset_df_4 = event_df3[(event_df3['time_value'] >= start_filter_4) & (event_df3['time_value'] <= end_filter_4)].copy()
subset_df_4[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
223753,2026-03-01 11:33:07,earthquake,9.0,9.0,8.0,NaN


In [19]:
# Fifth example: 44 p (43 used) and 40 s (81 used picks to locate and 84 total picks) event (within 2026-03-01 09:37:00 and 2026-03-01 09:38:00)
start_filter_5 = dt.datetime(2026, 3, 1, 9, 37, 0)
end_filter_5 = dt.datetime(2026, 3, 1, 9, 38, 0)
subset_df_5 = event_df3[(event_df3['time_value'] >= start_filter_5) & (event_df3['time_value'] <= end_filter_5)].copy()
subset_df_5[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
223728,2026-03-01 09:37:14,earthquake,84.0,81.0,43.0,NaN


### Seiscomp6

In [20]:
# Sixth example: 72 p and 72 s event (within 2026-06-01 04:59:00 and 2026-06-01 05:00:00) 130/144 used phases
start_filter_6 = dt.datetime(2026, 6, 1, 4, 59, 0)
end_filter_6 = dt.datetime(2026, 6, 1, 5, 0, 0)
subset_df_6 = event_df3[(event_df3['time_value'] >= start_filter_6) & (event_df3['time_value'] <= end_filter_6)].copy()
subset_df_6[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
245482,2026-06-01 04:59:07,earthquake,144.0,130.0,71.0,NaN


In [21]:
# Seventh example: 4 p and 4 s event (within 2026-06-01 00:25:00 and 2026-06-01 00:26:00)
start_filter_7 = dt.datetime(2026, 6, 1, 0, 25, 0)
end_filter_7 = dt.datetime(2026, 6, 1, 0, 26, 0)
subset_df_7 = event_df3[(event_df3['time_value'] >= start_filter_7) & (event_df3['time_value'] <= end_filter_7)].copy()
subset_df_7[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
245435,2026-06-01 00:25:08,earthquake,8.0,8.0,4.0,NaN


In [27]:
# Eighth example: 4 p and 3 s event (within 2026-06-01 07:50:00 and 2026-06-01 07:51:00)
start_filter_8 = dt.datetime(2026, 6, 1, 7, 50, 0)
end_filter_8 = dt.datetime(2026, 6, 1, 7, 50, 4)
subset_df_8 = event_df3[(event_df3['time_value'] >= start_filter_8) & (event_df3['time_value'] <= end_filter_8)].copy()
subset_df_8[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
245503,2026-06-01 07:50:04,earthquake,7.0,7.0,4.0,NaN


In [25]:
# Ninth example: 4 p and 2 s event (within 2026-06-03 07:03:00 and 2026-06-03 07:04:00)
start_filter_9 = dt.datetime(2026, 6, 3, 7, 3, 0)
end_filter_9 = dt.datetime(2026, 6, 3, 7, 4, 0)
subset_df_9 = event_df3[(event_df3['time_value'] >= start_filter_9) & (event_df3['time_value'] <= end_filter_9)].copy()
subset_df_9[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
245900,2026-06-03 07:03:12,earthquake,6.0,6.0,4.0,NaN


In [26]:
# Tenth example: 3 p and 3 s event (within 2026-06-03 15:22:00 and 2026-06-03 15:23:00)
start_filter_10 = dt.datetime(2026, 6, 3, 15, 22, 0)
end_filter_10 = dt.datetime(2026, 6, 3, 15, 23, 0)
subset_df_10 = event_df3[(event_df3['time_value'] >= start_filter_10) & (event_df3['time_value'] <= end_filter_10)].copy()
subset_df_10[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
245970,2026-06-03 15:22:29,not locatable,6.0,6.0,3.0,NaN
